# ASR-Join inspector — Chinese (zh-CN)

Inspect a single dataset's `asr_agg/rover/merged.jsonl` produced by
`asr_join` for Mandarin. Verifies:

1. **Schema is sane** — every row has `hypotheses.qwen`, `rover.text`, `rover.text_itn`.
2. **ITN actually fired** — count rows where `text_itn ≠ text` and where `text_itn` contains digits.
3. **Quality vs reference** — per-cut CER (char error rate, the right metric for Chinese) for qwen, rover, and ITN against `ref_text`.
4. **Audio + transcript browser** — slider over CER bins, play the original audio, see ref / qwen / itn side-by-side.

Default path: `/capstor/scratch/cscs/sgodey/quality_assesment_output/Result/quality_metrics/test_cv_zh_validation/asr_agg/rover/merged.jsonl` — change `MERGED_PATH` below to point elsewhere.

In [ ]:
# --- paths (edit if you point at a different dataset) -------------
MERGED_PATH = '/capstor/scratch/cscs/sgodey/quality_assesment_output/Result/quality_metrics/test_cv_zh_validation/asr_agg/rover/merged.jsonl'
SHAR_DIR    = '/capstor/scratch/cscs/sgodey/audio-datasets/SHAR/stage_2/commonvoice22_sidon/zh-CN/validation'   # for audio playback

import json, re, sys, os
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
pd.set_option('display.max_colwidth', 120)

## 1. Load `merged.jsonl` into a DataFrame

Flattens the nested structure: pulls `hypotheses.qwen.text` (Qwen's raw transcript), `rover.text` (post-ROVER consensus — identical to qwen when qwen is the only slot), `rover.text_itn` (after LLM normalization), and `ref_text` (the dataset's ground truth).

In [ ]:
def load_merged(path: str) -> pd.DataFrame:
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            r = json.loads(line)
            hyps = r.get('hypotheses') or {}
            rover = r.get('rover') or {}
            rows.append({
                'cut_id': r.get('cut_id'),
                'duration': r.get('duration'),
                'ref_text': (r.get('ref_text') or '').strip(),
                'qwen_text': ((hyps.get('qwen') or {}).get('text') or '').strip(),
                'rover_text': (rover.get('text') or '').strip(),
                'rover_text_itn': (rover.get('text_itn') or '').strip(),
                'filtered_reason': r.get('filtered_reason'),
                'lang_hint': r.get('language_hint'),
            })
    return pd.DataFrame(rows)

df = load_merged(MERGED_PATH)
print(f'loaded {len(df):,} rows from {MERGED_PATH}')
df.head(5)

## 2. ITN sanity — did the LLM actually do anything?

Two cheap checks:

- **`text_itn ≠ text`** — model produced *some* difference.
- **`text_itn` contains a digit** — the difference was actually a number normalization (not just a punctuation/whitespace jitter).

Then sample some rows where ITN actually fired so you can eyeball the conversion.

In [ ]:
has_diff = (df['rover_text'] != df['rover_text_itn']) & df['rover_text_itn'].str.len().gt(0)
has_digit_in_itn = df['rover_text_itn'].str.contains(r'\d', regex=True, na=False)
has_digit_in_raw = df['rover_text'].str.contains(r'\d', regex=True, na=False)

summary = pd.DataFrame({
    'count': [len(df), has_diff.sum(), has_digit_in_itn.sum(),
              (has_digit_in_itn & ~has_digit_in_raw).sum(),
              df['filtered_reason'].notna().sum()],
    'pct':   [100.0, 100*has_diff.mean(), 100*has_digit_in_itn.mean(),
              100*(has_digit_in_itn & ~has_digit_in_raw).mean(),
              100*df['filtered_reason'].notna().mean()],
}, index=[
    'total rows',
    'rows where text_itn ≠ text (LLM changed something)',
    'rows where text_itn contains a digit',
    'rows where ITN introduced digits not in raw (clean ITN win)',
    'rows with filtered_reason set',
])
summary.round(1)

In [ ]:
# Show 10 rows where ITN actually introduced digits — eyeball the conversions.
winners = df[has_diff & has_digit_in_itn & ~has_digit_in_raw].head(10)
print(f'Sample of {len(winners)} ITN "win" rows (digits appeared after normalization):')
for _, row in winners.iterrows():
    print(f'\n  cut={row["cut_id"]}')
    print(f'    ref     : {row["ref_text"]}')
    print(f'    qwen    : {row["qwen_text"]}')
    print(f'    text_itn: {row["rover_text_itn"]}')

## 3. CER against `ref_text`

Chinese is character-based — **CER**, not WER, is the right metric. Compute it for three hypotheses against the ground truth:

- `qwen_text` — Qwen3-ASR raw output
- `rover_text` — ROVER consensus (= qwen here since it's the only slot)
- `rover_text_itn` — after Gemma ITN

If `text_itn` CER < `qwen` CER → ITN improved quality (mostly digit alignment with ref). If CER is *higher* after ITN → the LLM is editorializing too much.

In [ ]:
import jiwer

def per_row_cer(refs: pd.Series, hyps: pd.Series) -> pd.Series:
    """Per-row CER. NaN when either side is empty."""
    out = pd.Series(np.nan, index=refs.index)
    mask = (refs.fillna('').str.len() > 0) & (hyps.fillna('').str.len() > 0)
    for i in refs.index[mask]:
        try:
            out.at[i] = jiwer.cer(refs.at[i], hyps.at[i])
        except Exception:
            pass
    return out

df['cer_qwen']     = per_row_cer(df['ref_text'], df['qwen_text'])
df['cer_rover']    = per_row_cer(df['ref_text'], df['rover_text'])
df['cer_rover_itn']= per_row_cer(df['ref_text'], df['rover_text_itn'])

agg = pd.DataFrame({
    'n_with_cer': [df['cer_qwen'].notna().sum(),
                   df['cer_rover'].notna().sum(),
                   df['cer_rover_itn'].notna().sum()],
    'mean_cer':   [df['cer_qwen'].mean(),
                   df['cer_rover'].mean(),
                   df['cer_rover_itn'].mean()],
    'median_cer': [df['cer_qwen'].median(),
                   df['cer_rover'].median(),
                   df['cer_rover_itn'].median()],
    'p90_cer':    [df['cer_qwen'].quantile(0.9),
                   df['cer_rover'].quantile(0.9),
                   df['cer_rover_itn'].quantile(0.9)],
}, index=['qwen', 'rover', 'rover_text_itn'])
agg.round(4)

In [ ]:
# CER distribution — overlaid histograms.
fig, ax = plt.subplots(figsize=(11, 5))
for col, color, lbl in [
    ('cer_qwen',      '#1f77b4', 'qwen raw'),
    ('cer_rover',     '#2ca02c', 'rover consensus'),
    ('cer_rover_itn', '#d62728', 'rover + ITN'),
]:
    vals = df[col].dropna()
    if len(vals) == 0: continue
    ax.hist(vals.clip(upper=1.5), bins=60, alpha=0.55,
            label=f'{lbl} (n={len(vals):,}, mean={vals.mean():.3f})',
            color=color, density=True, edgecolor='white', linewidth=0.3)
ax.set_xlabel('CER (clipped at 1.5)')
ax.set_ylabel('density')
ax.set_title('Per-cut CER distribution — zh-CN')
ax.legend()
ax.grid(True, axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Duration distribution — sanity check that the data is sensible.
fig, ax = plt.subplots(figsize=(11, 4))
ax.hist(df['duration'].dropna(), bins=80, color='#7f7f7f', edgecolor='white', linewidth=0.3)
ax.set_xlabel('duration (s)')
ax.set_ylabel('# cuts')
ax.set_title(f'Audio duration — total {df["duration"].sum()/3600:.1f} h')
ax.grid(True, axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## 4. ITN vs raw — direct delta

Per-row: did ITN improve or hurt CER vs the raw rover text? Histogram of `Δ = cer_rover_itn − cer_rover`:
- Negative = ITN better than raw
- Positive = ITN worse (LLM editorialized)
- Zero = no change (most rows — ITN only fires on rows with spoken numbers)

In [ ]:
delta = (df['cer_rover_itn'] - df['cer_rover']).dropna()
fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(delta.clip(-0.5, 0.5), bins=80, color='#9467bd', edgecolor='white', linewidth=0.3)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Δ CER = (ITN − raw)   — clipped to [-0.5, +0.5]')
ax.set_ylabel('# cuts')
n_better = (delta < -1e-6).sum()
n_worse  = (delta >  1e-6).sum()
n_same   = (delta.abs() <= 1e-6).sum()
ax.set_title(f'ITN-vs-raw CER delta: {n_better} better | {n_worse} worse | {n_same} unchanged')
ax.grid(True, axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## 5. Audio inspector

Slider over CER bins; for each bin shows random samples with `ref / qwen / rover / itn` side-by-side and plays the original audio. Helps diagnose what's actually breaking (silence, mishearing, code-switching, etc.).

**First run is slow** (builds an in-memory cut lookup from the SHAR). Subsequent slider moves are fast.

In [ ]:
# Make the legacy helper importable (build_cut_lookup wraps build_cutset cleanly).
_QME_DIR = Path('.').resolve()
while _QME_DIR.name != 'quality_metrics_enrichment' and _QME_DIR.parent != _QME_DIR:
    _QME_DIR = _QME_DIR.parent
if _QME_DIR.name == 'quality_metrics_enrichment':
    sys.path.insert(0, str(_QME_DIR / 'visualisation'))
from asr_quality_helper import build_cut_lookup

# Build lookup keyed on the cut_ids we actually have. Cached to disk so
# kernel restarts don't redo the work.
lookup = build_cut_lookup(
    shar_root=SHAR_DIR,
    keep_ids=set(df['cut_id']),
    target_sample_rate=16000,
    cache_path=Path('/tmp/asr_join_zh_cut_lookup.pkl'),
)
print(f'cut_lookup has {len(lookup):,} cuts')

In [ ]:
# Slider inspector — adapted from asr_quality_helper.metric_slider_inspector
# to show qwen + rover + itn instead of just rover (legacy shape was fused).
import html as _html
from IPython.display import Audio, HTML, clear_output, display
import ipywidgets as widgets

METRIC = 'cer_rover_itn'   # change to cer_qwen / cer_rover to browse those instead
N_BINS = 30
N_SAMPLES = 3
MAX_AUDIO_SEC = 12.0
SEED = None    # set int for reproducibility

valid = df[df[METRIC].notna()].reset_index(drop=True).copy()
arr = valid[METRIC].to_numpy()
quantiles = np.linspace(0, 1, N_BINS + 1)
edges = np.unique(np.quantile(arr, quantiles)).astype(float)
if edges.size < 2:
    edges = np.array([float(arr.min()), float(arr.min()) + 1e-9])
edges[-1] = np.nextafter(edges[-1], np.inf)
n_bins_actual = edges.size - 1

out = widgets.Output()

def _show(row):
    cid = row['cut_id']
    display(HTML(
        f"<code style='color:#444'>{_html.escape(cid)}</code> &nbsp;|&nbsp; "
        f"<b>{METRIC}</b>={row[METRIC]:.3f} &nbsp;|&nbsp; "
        f"dur={row.get('duration', float('nan')):.2f}s"
    ))
    def esc(s): return _html.escape((s or '')[:300])
    display(HTML(
        "<ul style='margin:2px 0'>"
        f"<li><b>ref</b>     : <span style='color:#111'>{esc(row['ref_text'])}</span></li>"
        f"<li><b>qwen</b>    : <span style='color:#111'>{esc(row['qwen_text'])}</span></li>"
        f"<li><b>rover</b>   : <span style='color:#111'>{esc(row['rover_text'])}</span></li>"
        f"<li><b>text_itn</b>: <span style='color:#c33'>{esc(row['rover_text_itn'])}</span></li>"
        "</ul>"
    ))
    cut = lookup.get(cid)
    if cut is None:
        display(HTML("<i>audio: cut not in lookup — skipping</i>")); return
    try:
        audio = cut.load_audio()
        sr = int(getattr(cut, 'sampling_rate'))
        data = np.asarray(audio)
        data = data[0] if data.ndim == 2 else np.squeeze(data)
        if MAX_AUDIO_SEC is not None and len(data) > int(MAX_AUDIO_SEC * sr):
            data = data[:int(MAX_AUDIO_SEC * sr)]
        display(Audio(data, rate=sr))
    except Exception as e:
        display(HTML(f"<i>audio load failed: {_html.escape(str(e))}</i>"))

def _refresh(b):
    b = max(0, min(int(b), n_bins_actual - 1))
    lo, hi = float(edges[b]), float(edges[b+1])
    in_bin = valid[(valid[METRIC] >= lo) & (valid[METRIC] < hi)]
    if in_bin.empty:
        d = (valid[METRIC] - lo).abs()
        in_bin = valid.loc[d.nsmallest(N_SAMPLES).index]
    drawn = in_bin.sample(n=min(N_SAMPLES, len(in_bin)),
                          random_state=SEED, replace=False)
    with out:
        clear_output(wait=True)
        display(HTML(
            f"<h4 style='margin:4px 0'>{METRIC} bin {b+1}/{n_bins_actual} "
            f"[{lo:.3f}, {hi:.3f}) &nbsp;— {len(in_bin):,} in bin, showing {len(drawn)}</h4>"
        ))
        for _, row in drawn.iterrows():
            _show(row); display(HTML("<hr style='border:none;border-top:1px solid #ddd'>"))
        display(reload_btn)

slider = widgets.IntSlider(value=n_bins_actual // 2, min=0, max=max(n_bins_actual-1, 0),
                            step=1, description=f'{METRIC} bin',
                            style={'description_width': 'initial'},
                            layout=widgets.Layout(width='80%'),
                            continuous_update=False)
reload_btn = widgets.Button(description='Reload', button_style='info', icon='refresh',
                            layout=widgets.Layout(width='120px'),
                            tooltip='Draw a new random sample from this bin')
slider.observe(lambda chg: _refresh(chg['new']), names='value')
reload_btn.on_click(lambda _: _refresh(slider.value))

display(widgets.VBox([slider, out]))
_refresh(slider.value)

## 6. ITN-winners audio browser

Filtered to **only the rows where ITN introduced a digit** that wasn't in the raw `rover.text` — i.e. the cases where Gemma actually did its job. Skips the (large) set of rows where ITN was a no-op so you spend your listening time on the interesting cuts.

Two buttons:
- **Next** / **Prev** — step through the filtered set in order.
- **Random** — jump to a random ITN winner.

Each card shows `ref / qwen / rover / itn` and plays the audio. Listen + read to judge whether the ITN-introduced digit matches what's actually being said.

In [ ]:
# Filter to ITN winners: text_itn introduced a digit not present in the raw rover text.
itn_winners = df[has_digit_in_itn & ~has_digit_in_raw].reset_index(drop=True).copy()
print(f'{len(itn_winners):,} ITN-winner rows out of {len(df):,} total')

if itn_winners.empty:
    print('No ITN winners — Gemma did not introduce digits anywhere. Check section 2 summary above.')
else:
    out_w = widgets.Output()
    state = {'idx': 0}

    def _render(i: int) -> None:
        i = max(0, min(int(i), len(itn_winners) - 1))
        state['idx'] = i
        # Update the index slider quietly to reflect the new position.
        # (Setting .value triggers .observe → which would call _render again;
        # we set state['idx'] first so the second call is a no-op anyway.)
        if counter.value != i:
            counter.value = i
        row = itn_winners.iloc[i]
        with out_w:
            clear_output(wait=True)
            display(HTML(
                f"<h4 style='margin:4px 0'>ITN winner {i+1} / {len(itn_winners):,}</h4>"
            ))
            _show(row)

    btn_prev = widgets.Button(description='← Prev',   icon='arrow-left',  layout=widgets.Layout(width='120px'))
    btn_next = widgets.Button(description='Next →',   icon='arrow-right', layout=widgets.Layout(width='120px'))
    btn_rand = widgets.Button(description='Random',   icon='random',     layout=widgets.Layout(width='120px'),
                              button_style='info')
    counter  = widgets.IntSlider(value=0, min=0, max=len(itn_winners)-1, step=1,
                                  description='index',
                                  style={'description_width': 'initial'},
                                  layout=widgets.Layout(width='60%'),
                                  continuous_update=False)

    btn_prev.on_click(lambda _: _render(state['idx'] - 1))
    btn_next.on_click(lambda _: _render(state['idx'] + 1))
    btn_rand.on_click(lambda _: _render(int(np.random.randint(len(itn_winners)))))
    counter.observe(lambda chg: _render(chg['new']), names='value')

    display(widgets.HBox([btn_prev, btn_next, btn_rand]))
    display(counter)
    display(out_w)
    _render(0)

## Tips

- **Browse only ITN winners**: filter the slider source with `df[has_digit_in_itn & ~has_digit_in_raw]` instead of `df`.
- **Compare metrics**: change `METRIC` in cell 5.2 to `cer_qwen` or `cer_rover` to slide along a different axis.
- **Different dataset**: edit `MERGED_PATH` + `SHAR_DIR` at the top and rerun. The notebook is dataset-agnostic — the only Chinese-specific choice is using CER (jiwer.cer) instead of WER.